In [1]:
import tensorflow as tf
import numpy as np

print(f"Versi TensorFlow: {tf.__version__}")
print(f"Versi NumPy: {np.__version__}")

# Tes GPU
gpu_list = tf.config.list_physical_devices('GPU')

if not gpu_list:
    print("=" * 40)
    print("!!! GAGAL !!!")
    print("TensorFlow TIDAK dapat menemukan GPU Anda.")
    print("=" * 40)
else:
    print("=" * 40)
    print("✨ BERHASIL! ✨")
    print("TensorFlow berhasil mendeteksi GPU Anda:")
    for gpu in gpu_list:
        print(f"- {gpu.name}")
    print("=" * 40)

# Tes operasi
try:
    with tf.device('/GPU:0'):
        a = tf.constant([[1.0, 2.0], [3.0, 4.0]])
        b = tf.constant([[1.0, 2.0], [3.0, 4.0]])
        c = tf.matmul(a, b)
    print("Operasi tes GPU berhasil dijalankan.")
    print(c.numpy())
except RuntimeError as e:
    print(f"Operasi tes GAGAL: {e}")

Versi TensorFlow: 2.20.0
Versi NumPy: 2.2.6
!!! GAGAL !!!
TensorFlow TIDAK dapat menemukan GPU Anda.
Operasi tes GPU berhasil dijalankan.
[[ 7. 10.]
 [15. 22.]]


In [3]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import numpy as np
import os
import time
import librosa.display
import matplotlib.pyplot as plt
import sounddevice as sd

print(f"TensorFlow Version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Batasi memori GPU agar tidak crash (opsional tapi disarankan)
        # tf.config.experimental.set_memory_growth(gpus[0], True) 
        print(f"Menggunakan GPU: {gpus[0].name}")
    except RuntimeError as e:
        print(e)
else:
    print("PERINGATAN: Tidak ada GPU ditemukan. Pelatihan akan SANGAT LAMBAT.")

# --- 1. Konfigurasi Utama ---
DATA_X_PATH = "./processed_data/X_train_deam.npy"
DATA_Y_PATH = "./processed_data/y_train_deam.npy"
CHECKPOINT_DIR = "./training_checkpoints"
OUTPUT_DIR = "./training_audio_samples"

# Konfigurasi Model
SAMPLE_RATE = 16000
SEGMENT_SEC = 2
SEGMENT_LENGTH = SAMPLE_RATE * SEGMENT_SEC # 32000
LATENT_DIM = 100  # Ukuran noise 'z'
EMOTION_DIM = 2   # Ukuran label emosi [V, A]

# Konfigurasi Pelatihan
# BATCH_SIZE SANGAT PENTING! 
# Jika GPU Anda 'Out of Memory', kecilkan angka ini (misal: 16 atau 8)
BATCH_SIZE = 32
EPOCHS = 500 # Mulai dengan 500, mungkin perlu 5000+ untuk hasil bagus
D_UPDATES_PER_G = 5  # Latih Diskriminator 5x lebih sering dari Generator
GP_WEIGHT = 10.0     # Bobot untuk Gradient Penalty (standar WGAN-GP)

# Buat folder jika belum ada
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- 2. Muat Data ---
print("Memuat dataset dari file .npy...")
try:
    X_data = np.load(DATA_X_PATH)
    y_data = np.load(DATA_Y_PATH)
    print(f"Data berhasil dimuat. X shape: {X_data.shape}, y shape: {y_data.shape}")
except FileNotFoundError:
    print("ERROR: File X_train_deam.npy atau y_train_deam.npy tidak ditemukan.")
    print("Pastikan Anda sudah menjalankan Fase 1.")
    exit()

# Konversi data ke float32 untuk TensorFlow
X_data = X_data.astype('float32')
y_data = y_data.astype('float32')

TensorFlow Version: 2.20.0
PERINGATAN: Tidak ada GPU ditemukan. Pelatihan akan SANGAT LAMBAT.
Memuat dataset dari file .npy...
Data berhasil dimuat. X shape: (32509, 32000, 1), y shape: (32509, 2)


In [ ]:
# --- Langkah 2: Membuat Pipa Data (tf.data Pipeline) ---
print("Membuat pipeline tf.data...")
# Buat dataset TensorFlow dari array NumPy
train_dataset = tf.data.Dataset.from_tensor_slices((X_data, y_data))

# Acak data, buat batch, dan atur prefetch
train_dataset = train_dataset.shuffle(buffer_size=X_data.shape[0]) \
                             .batch(BATCH_SIZE, drop_remainder=True) \
                             .prefetch(tf.data.AUTOTUNE)

print(f"Pipeline data siap. Jumlah steps per epoch: {len(train_dataset)}")


# --- Langkah 3: Arsitektur Model (Conditional WaveGAN) ---
from tensorflow.keras.layers import Input, Dense, Concatenate, Conv1D, Conv1DTranspose, LeakyReLU, Reshape, Flatten, BatchNormalization, Activation

# --- 3a. Generator ---
def build_generator(latent_dim, emotion_dim, output_len):
    z_input = Input(shape=(latent_dim,), name="z_noise")
    emotion_input = Input(shape=(emotion_dim,), name="emotion_label")

    combined_input = Concatenate()([z_input, emotion_input])
    
    # Mulai membangun waveform
    # (latent_dim + emotion_dim) -> 100 * 128
    x = Dense(100 * 128)(combined_input)
    x = Reshape((100, 128))(x)
    x = LeakyReLU(alpha=0.2)(x)
    
    # (100, 128) -> (400, 64)
    x = Conv1DTranspose(64, 25, strides=4, padding='same')(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.2)(x)
    
    # (400, 64) -> (2000, 32)
    x = Conv1DTranspose(32, 25, strides=5, padding='same')(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.2)(x)
    
    # (2000, 32) -> (8000, 16)
    x = Conv1DTranspose(16, 25, strides=4, padding='same')(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.2)(x)
    
    # (8000, 16) -> (32000, 1) -> Ini adalah output audio kita
    x = Conv1DTranspose(1, 25, strides=4, padding='same')(x)
    
    # Aktivasi Tanh menghasilkan output antara -1.0 dan 1.0
    audio_output = Activation('tanh', name="audio_output")(x)
    
    # Verifikasi shape: 100 * 4 * 5 * 4 * 4 = 32000. Sesuai.
    return Model(inputs=[z_input, emotion_input], outputs=audio_output, name="Generator")

# --- 3b. Discriminator ---
def build_discriminator(audio_len, emotion_dim):
    audio_input = Input(shape=(audio_len, 1), name="audio_segment")
    emotion_label_input = Input(shape=(emotion_dim,), name="emotion_label")

    # Logika untuk "menyuntikkan" label emosi
    e = Dense(audio_len)(emotion_label_input)
    e = Reshape((audio_len, 1))(e)
    combined_input = Concatenate(axis=2)([audio_input, e]) # Shape: (32000, 2)
    
    # Mulai membangun lapisan "kritikus"
    # (32000, 2) -> (8000, 16)
    x = Conv1D(16, 25, strides=4, padding='same')(combined_input)
    x = LeakyReLU(alpha=0.2)(x)
    
    # (8000, 16) -> (2000, 32)
    x = Conv1D(32, 25, strides=4, padding='same')(x)
    x = LeakyReLU(alpha=0.2)(x)
    
    # (2000, 32) -> (400, 64)
    x = Conv1D(64, 25, strides=5, padding='same')(x)
    x = LeakyReLU(alpha=0.2)(x)
    
    # (400, 64) -> (100, 128)
    x = Conv1D(128, 25, strides=4, padding='same')(x)
    x = LeakyReLU(alpha=0.2)(x)
    
    x = Flatten()(x)
    # Output TIDAK menggunakan sigmoid (ini adalah WGAN, bukan GAN biasa)
    output_score = Dense(1, name="output_score")(x) 
    
    return Model(inputs=[audio_input, emotion_label_input], outputs=output_score, name="Discriminator")

# Inisialisasi model
generator = build_generator(LATENT_DIM, EMOTION_DIM, SEGMENT_LENGTH)
discriminator = build_discriminator(SEGMENT_LENGTH, EMOTION_DIM)

print("--- Arsitektur Generator ---")
generator.summary()
print("\n--- Arsitektur Discriminator ---")
discriminator.summary()


# --- Langkah 4: Logika Pelatihan (WGAN-GP Custom Loop) ---

# --- 4a. Optimizers ---
generator_optimizer = Adam(learning_rate=1e-4, beta_1=0.5, beta_2=0.9)
discriminator_optimizer = Adam(learning_rate=1e-4, beta_1=0.5, beta_2=0.9)

# --- 4b. Fungsi Loss (Wasserstein Loss) ---
def discriminator_loss(real_output, fake_output):
    real_loss = tf.reduce_mean(real_output)
    fake_loss = tf.reduce_mean(fake_output)
    return fake_loss - real_loss

def generator_loss(fake_output):
    return -tf.reduce_mean(fake_output)

# --- 4c. Fungsi Gradient Penalty (Kunci WGAN-GP) ---
@tf.function
def gradient_penalty(batch_size, real_audio, fake_audio, real_labels):
    alpha = tf.random.normal([batch_size, 1, 1], 0.0, 1.0)
    interpolated = (alpha * real_audio) + ((1 - alpha) * fake_audio)

    with tf.GradientTape() as gp_tape:
        gp_tape.watch(interpolated)
        pred = discriminator([interpolated, real_labels], training=True)

    grads = gp_tape.gradient(pred, [interpolated])[0]
    norm = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=[1, 2]))
    gp = tf.reduce_mean((norm - 1.0) ** 2)
    return gp * GP_WEIGHT

# --- 4d. Checkpoint Manager ---
checkpoint = tf.train.Checkpoint(generator_optimizer=generator_optimizer,
                                 discriminator_optimizer=discriminator_optimizer,
                                 generator=generator,
                                 discriminator=discriminator)
ckpt_manager = tf.train.CheckpointManager(checkpoint, CHECKPOINT_DIR, max_to_keep=3)

if ckpt_manager.latest_checkpoint:
    checkpoint.restore(ckpt_manager.latest_checkpoint)
    print(f"Checkpoint dipulihkan dari {ckpt_manager.latest_checkpoint}")
else:
    print("Memulai pelatihan dari awal.")
    
# --- 4e. Fungsi train_step (Satu langkah pelatihan) ---
@tf.function
def train_step(real_audio, real_labels):
    noise = tf.random.normal([BATCH_SIZE, LATENT_DIM])
    
    # --- Latih Discriminator ---
    for _ in range(D_UPDATES_PER_G):
        with tf.GradientTape() as d_tape:
            fake_audio = generator([noise, real_labels], training=True)
            real_output = discriminator([real_audio, real_labels], training=True)
            fake_output = discriminator([fake_audio, real_labels], training=True)
            d_cost = discriminator_loss(real_output, fake_output)
            gp = gradient_penalty(BATCH_SIZE, real_audio, fake_audio, real_labels)
            d_loss = d_cost + gp

        d_gradients = d_tape.gradient(d_loss, discriminator.trainable_variables)
        discriminator_optimizer.apply_gradients(zip(d_gradients, discriminator.trainable_variables))

    # --- Latih Generator ---
    with tf.GradientTape() as g_tape:
        fake_audio = generator([noise, real_labels], training=True)
        fake_output = discriminator([fake_audio, real_labels], training=True)
        g_loss = generator_loss(fake_output)

    g_gradients = g_tape.gradient(g_loss, generator.trainable_variables)
    generator_optimizer.apply_gradients(zip(g_gradients, generator.trainable_variables))
    
    return d_loss, g_loss

# --- Langkah 5: Eksekusi Pelatihan (The Main Loop) ---

# --- 5a. Fungsi untuk monitoring ---
import soundfile as sf # Pastikan sudah `pip install soundfile`

seed_noise = tf.random.normal([4, LATENT_DIM])
seed_labels = tf.constant([
    [0.8, 0.7],  # Happy
    [-0.6, -0.5], # Sad
    [-0.7, 0.8],  # Angry
    [0.4, -0.3]  # Calm (Contoh V tinggi, A rendah)
], dtype=tf.float32)

def generate_and_save_samples(epoch, d_loss, g_loss):
    predictions = generator([seed_noise, seed_labels], training=False)
    
    print(
        f"Epoch {epoch+1} | "
        f"D Loss: {d_loss.numpy():.4f} | "
        f"G Loss: {g_loss.numpy():.4f}"
    )
    
    # Plot dan simpan waveform
    fig, axes = plt.subplots(4, 1, figsize=(10, 8))
    titles = ['Happy', 'Sad', 'Angry', 'Calm']
    for i in range(4):
        audio = predictions[i, :, 0].numpy()
        librosa.display.waveshow(audio, sr=SAMPLE_RATE, ax=axes[i])
        axes[i].set_title(titles[i])
        axes[i].set_ylim(-1.1, 1.1)
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"epoch_{epoch+1:04d}.png"))
    plt.show() # Tampilkan plot di notebook
    plt.close()
    
    # Simpan file audio 'Happy' dari epoch ini
    sf.write(os.path.join(OUTPUT_DIR, f"epoch_{epoch+1:04d}_happy.wav"), 
             predictions[0, :, 0].numpy(), 
             SAMPLE_RATE)

# --- 5b. The Main Training Loop ---
print("Memulai Pelatihan...")
print(f"Model akan dilatih selama {EPOCHS} epochs.")
print(f"Checkpoint akan disimpan di: {CHECKPOINT_DIR}")
print(f"Sampel audio akan disimpan di: {OUTPUT_DIR}")

start_time = time.time()

for epoch in range(EPOCHS):
    
    epoch_start_time = time.time()
    
    # Variabel untuk melacak loss rata-rata per epoch
    epoch_d_loss = []
    epoch_g_loss = []
    
    # 'train_dataset' sudah di-batch
    for step, (audio_batch, label_batch) in enumerate(train_dataset):
        d_loss, g_loss = train_step(audio_batch, label_batch)
        epoch_d_loss.append(d_loss.numpy())
        epoch_g_loss.append(g_loss.numpy())
        
    # --- Di akhir setiap epoch ---
    avg_d_loss = np.mean(epoch_d_loss)
    avg_g_loss = np.mean(epoch_g_loss)
    
    print(f"\n--- Akhir Epoch {epoch+1} ---")
    print(f"Waktu Epoch: {time.time() - epoch_start_time:.2f} detik")
    
    # Hasilkan sampel audio untuk monitoring
    generate_and_save_samples(epoch, avg_d_loss, avg_g_loss)
    
    # Simpan checkpoint setiap 20 epoch
    if (epoch + 1) % 20 == 0:
        ckpt_manager.save()
        print(f"Checkpoint untuk epoch {epoch+1} disimpan.")

print(f"\nPelatihan Selesai! Total waktu: {time.time() - start_time:.2f} detik")

# --- 5c. Simpan Model Generator Final ---
final_model_path = "generator_model.h5"
generator.save(final_model_path)
print(f"Model Generator final disimpan sebagai '{final_model_path}'")

Membuat pipeline tf.data...
Pipeline data siap. Jumlah steps per epoch: 1015
--- Arsitektur Generator ---


C:\Users\Auron\anaconda3\envs\venv_skripsi\lib\site-packages\keras\src\layers\activations\leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Model: "Generator"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ z_noise (InputLayer)          │ (None, 100)               │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ emotion_label (InputLayer)    │ (None, 2)                 │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ concatenate (Concatenate)     │ (None, 102)               │               0 │ z_noise[0][0],             │
│                               │                           │                 │ emotion_label[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense (Dense)                 │ (None, 12800)             │       1,318,400 │ concatenate[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ reshape (Reshape)             │ (None, 100, 128)          │               0 │ dense[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ leaky_re_lu (LeakyReLU)       │ (None, 100, 128)          │               0 │ reshape[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1d_transpose              │ (None, 400, 64)           │         204,864 │ leaky_re_lu[0][0]          │
│ (Conv1DTranspose)             │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization           │ (None, 400, 64)           │             256 │ conv1d_transpose[0][0]     │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ leaky_re_lu_1 (LeakyReLU)     │ (None, 400, 64)           │               0 │ batch_normalization[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1d_transpose_1            │ (None, 2000, 32)          │          51,232 │ leaky_re_lu_1[0][0]        │
│ (Conv1DTranspose)             │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_1         │ (None, 2000, 32)          │             128 │ conv1d_transpose_1[0][0]   │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ leaky_re_lu_2 (LeakyReLU)     │ (None, 2000, 32)          │               0 │ batch_normalization_1[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1d_transpose_2            │ (None, 8000, 16)          │          12,816 │ leaky_re_lu_2[0][0]        │
│ (Conv1DTranspose)             │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_2         │ (None, 8000, 16)          │              64 │ conv1d_transpose_2[0][0]   │
│ (BatchNormalization)          │                           │               

 Total params: 1,588,161 (6.06 MB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ emotion_label (InputLayer)    │ (None, 2)                 │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_1 (Dense)               │ (None, 32000)             │          96,000 │ emotion_label[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ audio_segment (InputLayer)    │ (None, 32000, 1)          │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ reshape_1 (Reshape)           │ (None, 32000, 1)          │               0 │ dense_1[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ concatenate_1 (Concatenate)   │ (None, 32000, 2)          │               0 │ audio_segment[0][0],       │
│                               │                           │                 │ reshape_1[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1d (Conv1D)               │ (None, 8000, 16)          │             816 │ concatenate_1[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ leaky_re_lu_4 (LeakyReLU)     │ (None, 8000, 16)          │               0 │ conv1d[0][0]               │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1d_1 (Conv1D)             │ (None, 2000, 32)          │          12,832 │ leaky_re_lu_4[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ leaky_re_lu_5 (LeakyReLU)     │ (None, 2000, 32)          │               0 │ conv1d_1[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1d_2 (Conv1D)             │ (None, 400, 64)           │          51,264 │ leaky_re_lu_5[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ leaky_re_lu_6 (LeakyReLU)     │ (None, 400, 64)           │               0 │ conv1d_2[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1d_3 (Conv1D)             │ (None, 100, 128)          │         204,928 │ leaky_re_lu_6[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ leaky_re_lu_7 (LeakyReLU)     │ (None, 100, 128)          │               0 │ conv1d_3[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ flatten (Flatten)             │ (None, 12800)             │               0 │ leaky_re_lu_7[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ output_score (Dense)          │ (None, 1)                 │          12,801 │ flatten[0][0]              │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 378,641 (1.44 MB)

 Trainable params: 378,641 (1.44 MB)

 Non-trainable params: 0 (0.00 B)

Memulai pelatihan dari awal.
Memulai Pelatihan...
Model akan dilatih selama 500 epochs.
Checkpoint akan disimpan di: ./training_checkpoints
Sampel audio akan disimpan di: ./training_audio_samples
